In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import *
from src.io import *
from src.plotting import *
from src.correction import *
from src.template_matching import *
from scripts import preprocess_01, template_matching_02
from pathlib import Path

import hyperspy.api as hs

%matplotlib qt5

#### Working code to obtain the results for my thesis paper. The cells are organized in a way that they can be run sequentially to reproduce the results which are shown in the paper. The results are saved in the `results` folder. The code is written in Python and used the Hyperspy bundle package.

In [ ]:
preprocess_01.Al_preprocess()

In [ ]:
template_matching_02.Al_template_matching()

In [ ]:
sim = Sim(angular_resolution=0.5, precession_angle=1.0, min_intensity=1e-5, atom_type="Al", lattice_constant=4.05, space_group=225)

result_area_I = load_results(PROJECT_ROOT / "results" / "Al" /  "masked_area_I_results.pkl", sim)
result_area_II = load_results(PROJECT_ROOT / "results" / "Al" /  "masked_area_II_results.pkl", sim)

corrected_result_area_I, line_area_I = correction_algorithm(PROJECT_ROOT / "results" / "Al" /  "masked_area_I_results.pkl", sim, n_iter=2000, threshold=0.05, min_inliers=10)
corrected_result_area_II, line_area_II = correction_algorithm(PROJECT_ROOT / "results" / "Al" /  "masked_area_II_results.pkl", sim, n_iter=2000, threshold=0.05, min_inliers=10)

In [ ]:
tilt_orientations_area_I = convert_best_match_to_orientations(result_area_I)
tilt_orientations_area_II = convert_best_match_to_orientations(result_area_II)

corrected_tilt_orientations_area_I = convert_best_match_to_orientations(corrected_result_area_I)
corrected_tilt_orientations_area_II = convert_best_match_to_orientations(corrected_result_area_II)

line_orientations_area_I = line_to_orientations(line_area_I, sim.phase.point_group)
line_orientations_area_II = line_to_orientations(line_area_II, sim.phase.point_group)

### Visualization

In [ ]:
tilt_ipf(tilt_orientations_area_I[:, 0], sim, line_orientations_area_I)
tilt_ipf(tilt_orientations_area_II[:, 0], sim, line_orientations_area_II)

In [ ]:
tilt_ipf(corrected_tilt_orientations_area_I[:, 0], sim, line_orientations_area_I)
tilt_ipf(corrected_tilt_orientations_area_II[:, 0], sim, line_orientations_area_II)

In [ ]:
NCC_IPF(result_area_I, 59, Path("D:\\Datasets\\aluminium_tilt_series\\preprocessed\\mean_area_I"))
NCC_IPF(result_area_II, 59, Path("D:\\Datasets\\aluminium_tilt_series\\preprocessed\\mean_area_II"))


In [ ]:
NCC_IPF(corrected_result_area_I, 0, Path("D:\\Datasets\\aluminium_tilt_series\\preprocessed\\mean_area_I"))
NCC_IPF(corrected_result_area_II, 0, Path("D:\\Datasets\\aluminium_tilt_series\\preprocessed\\mean_area_II"))

In [ ]:
misorientation = tilt_orientations_area_II[:, 0] - tilt_orientations_area_I[:, 0]
corrected_misorientation = corrected_tilt_orientations_area_II[:, 0] - corrected_tilt_orientations_area_I[:, 0]

In [ ]:
misorientation.scatter('homochoric')

In [ ]:
plt.hist(np.rad2deg(misorientation.angle), bins=60)

In [ ]:
plot_misorientation(misorientation.angle)